# Experiment 2: Production Rules and the Match Phase

Represent domain knowledge using IF–THEN production rules, maintain working memory, and compute the conflict set of applicable rules.

## Algorithm

1. Encode each rule with an identifier, conditions, and conclusion.
2. Initialise working memory as a set of facts.
3. A rule matches when all its conditions are in working memory and its conclusion is absent.
4. Collect all matching rules into the conflict set.

In [1]:
rules = [
    {"id": "R1", "conditions": {"fever", "cough"}, "conclusion": "flu"},
    {"id": "R2", "conditions": {"flu"}, "conclusion": "rest"},
    {"id": "R3", "conditions": {"fever"}, "conclusion": "check_temperature"},
    {"id": "R4", "conditions": {"cough", "sore_throat"}, "conclusion": "cold"}
]

def match_rules(rules, wm):
    conflict_set = []
    for rule in rules:
        if rule["conditions"].issubset(wm) and rule["conclusion"] not in wm:
            conflict_set.append(rule["id"])
    return conflict_set

In [2]:
wm = {"fever", "cough"}
conflict_set = match_rules(rules, wm)
print("Working Memory:", wm)
print("Conflict Set:", conflict_set)

Working Memory: {'fever', 'cough'}
Conflict Set: ['R1', 'R3']


## Exercise 1: Add facts to WM one at a time

In [3]:
wm = set()
facts_to_add = ["fever", "cough", "sore_throat", "flu"]

for fact in facts_to_add:
    wm.add(fact)
    print(f"Added: {fact}")
    print("WM:", wm)
    print("Conflict Set:", match_rules(rules, wm))
    print()

Added: fever
WM: {'fever'}
Conflict Set: ['R3']

Added: cough
WM: {'fever', 'cough'}
Conflict Set: ['R1', 'R3']

Added: sore_throat
WM: {'fever', 'cough', 'sore_throat'}
Conflict Set: ['R1', 'R3', 'R4']

Added: flu
WM: {'flu', 'fever', 'cough', 'sore_throat'}
Conflict Set: ['R2', 'R3', 'R4']



## Exercise 2: Negated conditions

In [4]:
rules_with_negation = [
    {"id": "R1", "conditions": {"fever", "cough"}, "conclusion": "flu"},
    {"id": "R2", "conditions": {"flu"}, "conclusion": "rest"},
    {"id": "R3", "conditions": {"fever"}, "conclusion": "check_temperature"},
    {"id": "R4", "conditions": {"cough", "NOT:sore_throat"}, "conclusion": "possible_bronchitis"}
]

def condition_matches(condition, wm):
    if condition.startswith("NOT:"):
        return condition[4:] not in wm
    return condition in wm

def match_rules_with_negation(rules, wm):
    conflict_set = []
    for rule in rules:
        conditions_match = all(condition_matches(c, wm) for c in rule["conditions"])
        if conditions_match and rule["conclusion"] not in wm:
            conflict_set.append(rule["id"])
    return conflict_set


In [5]:
wm = {"cough"}
print("WM:", wm)
print("Conflict Set:", match_rules_with_negation(rules_with_negation, wm))

wm.add("sore_throat")
print("\nAfter adding sore_throat:")
print("WM:", wm)
print("Conflict Set:", match_rules_with_negation(rules_with_negation, wm))

WM: {'cough'}
Conflict Set: ['R4']

After adding sore_throat:
WM: {'cough', 'sore_throat'}
Conflict Set: []


## Exercise 3: Match time for 1,000 auto-generated rules

In [9]:
import time

for count in range(1,100000,10000):
    auto_rules = []
    for i in range(count):
        auto_rules.append({
            "id": f"R{i}",
            "conditions": {f"fact{i}", f"fact{i+1}"},
            "conclusion": f"result{i}"
        })

    wm = {f"fact{i}" for i in range(1001)}

    start = time.perf_counter()
    conflict_set = match_rules(auto_rules, wm)
    elapsed = time.perf_counter() - start

    print("\nNumber of rules:", len(auto_rules))
    print("Conflict set size:", len(conflict_set))
    print(f"Match time: {elapsed:.8f} seconds")


Number of rules: 1
Conflict set size: 1
Match time: 0.00014460 seconds

Number of rules: 10001
Conflict set size: 1000
Match time: 0.00247080 seconds

Number of rules: 20001
Conflict set size: 1000
Match time: 0.00279250 seconds

Number of rules: 30001
Conflict set size: 1000
Match time: 0.00385510 seconds

Number of rules: 40001
Conflict set size: 1000
Match time: 0.00652380 seconds

Number of rules: 50001
Conflict set size: 1000
Match time: 0.00649570 seconds

Number of rules: 60001
Conflict set size: 1000
Match time: 0.00761360 seconds

Number of rules: 70001
Conflict set size: 1000
Match time: 0.00963740 seconds

Number of rules: 80001
Conflict set size: 1000
Match time: 0.01010280 seconds

Number of rules: 90001
Conflict set size: 1000
Match time: 0.01365720 seconds


As the number of rules increases, the matcher checks more conditions, so match time increases. RETE was developed to avoid repeatedly rechecking unchanged conditions by storing and reusing partial matches.